In [1]:
import re
from konlpy.tag import Okt, Kkma
import pandas as pd

In [ ]:
# 정규 표현식 연습
raw_text = "안녕하세요!!! 123반갑습니다... @@@자연어 처리는 재미있어요 ^_^"
# 한글과 공백을 제외하고 모두 제거
cleaned_text = re.sub(r'[^가-힣\s]', '', raw_text)


In [3]:
# 한국어 형태소 분석기 비교
okt = Okt()
kkma = Kkma()
text = '나는 사과를 먹는다.'
print(okt.pos(text))
print(kkma.pos(text))

[('나', 'Noun'), ('는', 'Josa'), ('사과', 'Noun'), ('를', 'Josa'), ('먹는다', 'Verb'), ('.', 'Punctuation')]
[('나', 'VV'), ('는', 'ETD'), ('사과', 'NNG'), ('를', 'JKO'), ('먹', 'VV'), ('는', 'EPT'), ('다', 'EFN'), ('.', 'SF')]


In [ ]:
# okt 형태소 분석기를 이용해서 조사와 구두점을 제거
text = '사과가 맛있다, 나는 사과를 먹는다'
stopwords = ['Josa', 'Punctuation']
print(okt.pos(text))
cleaned_text = [t for t, pos in okt.pos(text) if pos not in stopwords]
cleaned_text

    

[('사과', 'Noun'), ('가', 'Josa'), ('맛있다', 'Adjective'), (',', 'Punctuation'), ('나', 'Noun'), ('는', 'Josa'), ('사과', 'Noun'), ('를', 'Josa'), ('먹는다', 'Verb')]


['사과', '맛있다', '나', '사과', '먹는다']

In [ ]:
# 다음 영화리뷰 데이터셋
# 1. 데이터 로드
# 2. 전처리 (정규식 이용해서 한글과 공백만 추출)
# 3. 형태소 분석 및 품사 필터링 (명사 동사 형용사만 추출)
# 4. TTR 계산 (상위 100개 리뷰를 대상으로 전체 토큰 수 대비 고유 타입의 수를 비율(TTR)로 계산)
    # TTR < 0.5 어휘 반복이 많아서 TTR이 낮게 측정 Else 다양한 어휘가 사용되서 TTR 높게 측정
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
text = list(df['review'])
cleaned_text = [re.sub(r'[^가-힣\s]', '', doc) for doc in text]

POS = ['Noun', 'Verb', 'Adjective']
okt = Okt()

tokenized = []
for i in cleaned_text:
    doc = [t for t, pos in okt.pos(i) if pos in POS]
    tokenized.append(doc)

돈 들인건 티가 나지만 보는 내내 하품만


In [6]:
all_tokens = [ token for tokens in tokenized for token in tokens]

In [15]:
V = list ( set( token for tokens in tokenized for token in tokens ) )

TTR = len(V) / len(all_tokens)
TTR

0.13766003555491801

In [22]:
# 1. 데이터로드
import pandas as pd
daum_df= pd.read_csv('daum_movie_review.csv')
corpus = daum_df['review'].to_numpy()
corpus

array(['돈 들인건 티가 나지만 보는 내내 하품만',
       '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.',
       '이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌... 그래도 다음 작품을 기대하게 만든다...',
       ..., '가족을 위한 영화... 괜찮은 영화.~~~',
       '간만에 제대로 잘짜여진 각본의 영화를 봤네 여운이 아직도 남아~어른을 위한 애니~',
       '한국개봉을 눈빠지게 기다린 보람이있다 깨우치는게 많은 영화'], shape=(14725,), dtype=object)

In [23]:
# 2. 전처리(정규식이용해서 한글과 공백만 추출)
import re
cleaned_corpus = [re.sub(r'[^가-힣\s]','',doc)  for doc in corpus]
cleaned_corpus[:5]

['돈 들인건 티가 나지만 보는 내내 하품만',
 '몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남',
 '이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌 그래도 다음 작품을 기대하게 만든다',
 '이 정도면 볼만하다고 할 수 있음',
 '재미있다']

In [24]:
# 3. 형태소 분석 및 품사 필터링(명사 동사 형용사만 추출)
def pos_noun_verb_ajective(doc):
    okt = Okt()
    temp = []
    for token , pos in okt.pos(doc):
        if pos in ['Noun','Verb','Adjective'] and len(token)>=2:
            temp.append(token)
    return temp

pos_noun_verb_ajective('.....')


[]

In [25]:
# 3. 형태소 분석 및 품사 필터링(명사 동사 형용사만 추출)
okt = Okt()
cleaned_corpus_pos = []
for copors in cleaned_corpus:
    temp = pos_noun_verb_ajective(copors)
    if len(temp) > 0:
        cleaned_corpus_pos.append(temp)  

In [26]:
# 4. TTR 계산  (상위 100개 리뷰를 대상으로 전체 토큰수 대비 고유타입의 수를 비율(TTR) 계산)
    #   TTR < 0.5 어휘반복이 많아서 TTR이 낮게 측정  else  다양한 어휘가 사용되어서 TTR이 높게 책성

# 전체토큰

#토큰수 기준 상위 100
import numpy as np
top100_index = np.argsort([-len(doc) for doc in cleaned_corpus_pos])[:100]

top100_corpus = [cleaned_corpus_pos[index] for index in top100_index]

len(set(top100_corpus[0])) / len(top100_corpus[0])

0.8285714285714286

In [ ]:
# 평점예측모델
# 1. 데이터 로드
# 2. 전처리
# 3. 벡터화
# 4. 차원축소(옵션)
# 5. 모델선택
# 6. 학습
# 7. 평가
# 8. 배포- 서비스(AWS)